# 🧬 Biomolecular splitters: protein-axis holdouts

Welcome! The `biomolecular` family holds out data along a **protein/biomolecule axis** rather than a purely chemical one: sequence identity, target family, binding-site similarity, structure deposition date, and a joint ligand+sequence axis. Reach for these whenever the generalisation claim you actually care about is "a new target" or "a new complex," not just "a new molecule" — a plain scaffold or similarity split on the ligand side alone will silently leak protein-level information. All five classes below live in `chemsplit.splitters.biomolecular`.

**Contents**
1. [🧬 SequenceIdentitySplitter](#1)
2. [🗂️ ProteinFamilySplitter](#2)
3. [🧪 BindingSiteSplitter](#3)
4. [📅 DepositionDateSplitter](#4)
5. [🔗 ComplexJointSplitter](#5)

Shared setup: silence RDKit's standardizer logging, import the family, and define a tiny result-summary helper reused throughout.

In [1]:
import numpy as np
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")

from chemsplit.datasets import make_dated_series, make_scaffold_families, make_sequences
from chemsplit.splitters.biomolecular import (
    BindingSiteSplitter,
    ComplexJointSplitter,
    DepositionDateSplitter,
    ProteinFamilySplitter,
    SequenceIdentitySplitter,
)


def summarize(name, result):
    print(f"{name}: n={result.n_records} train={len(result.train)} "
          f"valid={len(result.valid)} test={len(result.test)} discard={len(result.discard)}")

<a id="1"></a>
## 1. 🧬 SequenceIdentitySplitter

Groups protein sequences by pairwise identity (single-linkage above a threshold), then holds out whole groups so no near-identical sequence pair straddles train/test — the standard control for any model that takes a protein as input.

| Parameter | Meaning |
|---|---|
| `identity_threshold` | sequences whose pairwise identity exceeds this are merged into one group (single-linkage); default `0.7` |
| `algorithm` | `'parasail'` (global alignment, `bio` extra), `'hamming'` (dependency-free fractional-match fallback), or `'auto'` (parasail if installed, else hamming); default `'auto'` |

> 💡 **Advantages**
> - The standard control for any model taking a protein as input — without it, a "new target" claim isn't credible.
> - Computes and reports `max_cross_identity`, so the claim is backed by evidence.
> - Makes the `identity_denominator` choice explicit, removing the most common source of incomparable identity numbers between papers.
> - Deduplicating to unique sequences first keeps it cheap even on large interaction datasets.

> ⚠️ **Pitfalls**
> - **Percent identity isn't one number.** The same alignment gives very different identities depending on the denominator and on local vs. global alignment.
> - A poor proxy for binding-site similarity — two proteins at 20% overall identity can share nearly identical pockets. Pair with `binding_site`.
> - `greedy_incremental` is order-dependent by construction; longest-first fixes it here, but results won't match other tools' orderings.
> - Multi-domain and multi-chain proteins align poorly as single strings; coverage filtering helps but doesn't solve it.

In [2]:
fx = make_sequences(n=40, families=6, identity_within=0.85, seed=0)
sp = SequenceIdentitySplitter(identity_threshold=0.5, algorithm="hamming",
                               train_size=0.7, test_size=0.3, random_state=0)
result = sp.split_result(fx.sequences, X_kind="sequences")[0]
summarize("SequenceIdentitySplitter", result)

SequenceIdentitySplitter: n=36 train=24 valid=0 test=12 discard=0


`algorithm="hamming"` above is a dependency-free equal-length fractional-match fallback — a weak proxy for real sequence identity. With the `bio` extra installed, `algorithm="parasail"` runs a proper global alignment instead, and handles sequences of different lengths.

In [3]:
sp_aligned = SequenceIdentitySplitter(identity_threshold=0.5, algorithm="parasail",
                                       train_size=0.7, test_size=0.3, random_state=0)
result_aligned = sp_aligned.split_result(fx.sequences, X_kind="sequences")[0]
summarize("SequenceIdentitySplitter (parasail)", result_aligned)

SequenceIdentitySplitter (parasail): n=36 train=24 valid=0 test=12 discard=0


<a id="2"></a>
## 2. 🗂️ ProteinFamilySplitter

Holds out whole target families/classes by a caller-supplied hierarchy (e.g. kinase family labels) — a coarser, label-driven alternative to sequence-identity clustering, with no external database lookup: the family label per record is supplied directly by the caller.

| Parameter | Meaning |
|---|---|
| `family_labels` | one family/class label per record, aligned with `X`; required, no default inference |

> 💡 **Advantages**
> - Tests transfer *across target classes* — kinases in train, GPCRs in test — invisible to a sequence-identity split within one family.
> - Uses curated biological knowledge instead of a sequence heuristic, so the groups mean something to a biologist.
> - Explicit `held_out_families` makes the experiment fully specifiable in one line.

> ⚠️ **Pitfalls**
> - Family annotations are incomplete and inconsistent; unlabelled targets form a junk group whose size must be checked.
> - Family boundaries don't imply pharmacological independence — kinase and non-kinase ATP-binding proteins share ligand chemistry.
> - Most datasets have very few families, so holding one out is high-variance; prefer leave-one-family-out via `leave_one_cluster_out`.
> - Family sizes are extremely skewed (kinases dominate public data), so the requested ratio is usually unreachable.

In [4]:
seqs = [f"SEQ{i}" for i in range(30)]
family_labels = [f"family_{i % 6}" for i in range(30)]
sp = ProteinFamilySplitter(family_labels=family_labels, train_size=0.7, test_size=0.3, random_state=0)
result = sp.split_result(seqs, X_kind="sequences")[0]
summarize("ProteinFamilySplitter", result)

ProteinFamilySplitter: n=30 train=20 valid=0 test=10 discard=0


<a id="3"></a>
## 3. 🧪 BindingSiteSplitter

Clusters on pocket residue composition/sequence rather than global sequence identity — useful when two proteins share a binding pocket despite low overall sequence identity, catching a leak that sequence identity alone misses.

| Parameter | Meaning |
|---|---|
| `representation` | `'composition'` clusters a caller-supplied residue-composition feature matrix via Butina on Euclidean distance; `'pocket_sequence'` clusters short pocket sequences via the identity machinery; default `'composition'` |
| `cutoff` | Butina cutoff (distance for `'composition'`, `1 - identity` for `'pocket_sequence'`); default `0.35` |

> 💡 **Advantages**
> - Catches the leak sequence identity misses: distant sequences with near-identical pockets, common across convergently evolved binding sites.
> - Pocket composition is directly interpretable — you can read off which residues drive the grouping.
> - Works from any pocket definition, including one derived from a predicted structure.

> ⚠️ **Pitfalls**
> - **Needs a pocket definition the library can't produce.** Pocket detection is its own research problem, and a different detector yields a different split.
> - `residue_composition` ignores geometry entirely — two pockets with identical residue counts and completely different shapes look identical to it.
> - Pocket residue lists from a single co-crystal reflect one ligand's contacts, not the pocket itself.
> - Unavailable for targets without structures — missing pockets raise rather than get silently dropped.

Below, two tight synthetic composition blobs are deliberately used to keep the demo fast — Butina then finds exactly two clusters, whose sizes don't line up with the requested 70/30 split, so the honest `SizeToleranceWarning` below is expected, not a bug.

In [5]:
rng = np.random.default_rng(0)
blob_a = rng.normal(loc=0.0, scale=0.05, size=(20, 4))
blob_b = rng.normal(loc=5.0, scale=0.05, size=(20, 4))
F = np.vstack([blob_a, blob_b])
sp = BindingSiteSplitter(representation="composition", cutoff=0.5,
                          train_size=0.7, test_size=0.3, random_state=0)
result = sp.split_result(F, X_kind="features")[0]
summarize("BindingSiteSplitter", result)

BindingSiteSplitter: n=40 train=20 valid=0 test=20 discard=0


<a id="4"></a>
## 4. 📅 DepositionDateSplitter

Specializes the `temporal` date-cut for deposited structures: after the ordinary temporal cut, any *train* record whose ligand Tanimoto similarity or sequence identity to a *test* record exceeds a ceiling is additionally pruned to `discard` — closing the leak a pure date cut leaves open, since the same ligand series and protein get redeposited for years.

| Parameter | Meaning |
|---|---|
| `cut_date` | records with `dates <= cut_date` are candidate train; later records are candidate test |
| `ligand_similarity_ceiling` | if given, train ligands more similar than this (ECFP4 Tanimoto) to any test ligand are pruned; default `None` |
| `sequence_identity_ceiling` | if given, train sequences more identical than this to any test sequence are pruned; default `None` |

> 💡 **Advantages**
> - The established protocol for evaluating docking, scoring functions, and co-folding — and the reason several early scoring-function results failed to reproduce.
> - The added ligand and sequence pruning closes the leak a pure date cut leaves open.
> - Every pruning decision is counted and reported.

> ⚠️ **Pitfalls**
> - A deposition-date cut alone is a weak split — redundant re-depositions of the same complex put near-identical entries on both sides of the cut.
> - Deposition date isn't discovery date; structures are often deposited long after the work.
> - Pruning by ligand similarity removes exactly the complexes most informative for testing, which depresses absolute scores intentionally.
> - Sequence pruning is off by default because it needs sequences; leaving it off keeps homologous complexes in training.

In [6]:
fx = make_dated_series(n=100, seed=0)
median_date = str(np.sort(fx.dates)[len(fx.dates) // 2])
sp = DepositionDateSplitter(cut_date=median_date, random_state=0)
result = sp.split_result(fx.smiles, dates=fx.dates)[0]
summarize("DepositionDateSplitter", result)
print("cut_date:", sp.cut_date, "| n_pruned:", result.metadata.get("n_pruned"))

DepositionDateSplitter: n=100 train=51 valid=0 test=49 discard=0
cut_date: 2017-06-24 | n_pruned: 0


The date cut alone barely separates anything if the same ligand series gets redeposited for years — `ligand_similarity_ceiling` additionally prunes any train ligand that is too Tanimoto-similar to a test ligand (and `sequence_identity_ceiling` does the same on the protein axis), moving it to `discard` instead of leaving a near-duplicate in train.

In [7]:
seqs_dated = make_sequences(n=100, families=10, identity_within=0.9, seed=0).sequences
sp_ceil = DepositionDateSplitter(
    cut_date=median_date, ligand_similarity_ceiling=0.6, sequence_identity_ceiling=0.7, random_state=0,
)
result_ceil = sp_ceil.split_result(fx.smiles, dates=fx.dates, sequences=seqs_dated)[0]
summarize("DepositionDateSplitter (with ceilings)", result_ceil)
print("n_pruned:", result_ceil.metadata["n_pruned"], "| discard:", len(result_ceil.discard))

DepositionDateSplitter (with ceilings): n=100 train=50 valid=0 test=49 discard=1
n_pruned: 1 | discard: 1


<a id="5"></a>
## 5. 🔗 ComplexJointSplitter

The strictest protocol in the family: jointly novel on the ligand-similarity axis **and** the sequence-identity axis. Composes two independent group axes through `chemsplit._pair_assign.assign_pair_groups`, so a test complex is guaranteed novel on both axes (`mode="both_novel"`) or at least one axis (`mode="either_novel"`).

| Parameter | Meaning |
|---|---|
| `ligand_grouper` | groups records by ligand similarity; `None` falls back to Butina clustering directly; default `None` |
| `sequence_grouper` | groups records by sequence identity; `None` falls back to this module's own `SequenceIdentitySplitter`; default `None` |
| `mode` | `'both_novel'` (strictest) or `'either_novel'` (looser, larger surviving test set); default `'both_novel'` |

> 💡 **Advantages**
> - The only defensible setting for claiming generalisation to genuinely new complexes — both constraints are verified and reported.
> - `mode="either_novel"` gives an intermediate, larger-data experiment when `"both_novel"` leaves too little.

> ⚠️ **Pitfalls**
> - Leaves very little data — on typical structural datasets the discard fraction exceeds 80%. Reports the fraction and refuses beyond `max_discard_frac`.
> - Two thresholds and two groupers compound into four choices defining the experiment, none with a canonical value.
> - A tiny test set invites over-interpreting a single number — report per-complex results, not just an aggregate.
> - `mode="either_novel"` is much weaker and is frequently reported as if it were `"both_novel"`.

Note the large `discard` count below: this dataset uses `n_scaffolds=10` correlated with `families=10`, so most complexes fail to be simultaneously novel on both axes — exactly the strictness the pitfalls above warn about.

In [8]:
fx = make_scaffold_families(n_scaffolds=10, per_scaffold=10, seed=0)
seqs = make_sequences(n=100, families=10, identity_within=0.9, seed=0).sequences
sp = ComplexJointSplitter(mode="both_novel", random_state=0, train_size=0.8, test_size=0.2)
result = sp.split_result(fx.smiles, sequences=seqs)[0]
summarize("ComplexJointSplitter", result)

ComplexJointSplitter: n=100 train=34 valid=0 test=19 discard=47


`ligand_grouper`/`sequence_grouper` default to an internal Butina/`SequenceIdentitySplitter` fallback — pass explicit instances to control the thresholds that define "novel" on each axis.

In [9]:
from chemsplit.splitters.similarity import ButinaSplitter

sp_explicit = ComplexJointSplitter(
    ligand_grouper=ButinaSplitter(cutoff=0.3, random_state=0),
    sequence_grouper=SequenceIdentitySplitter(identity_threshold=0.6),
    mode="both_novel", random_state=0, train_size=0.8, test_size=0.2,
)
result_explicit = sp_explicit.split_result(fx.smiles, sequences=seqs)[0]
summarize("ComplexJointSplitter (explicit groupers)", result_explicit)

ComplexJointSplitter (explicit groupers): n=100 train=36 valid=0 test=21 discard=43


That covers the `biomolecular` family. Pair a sequence- or family-level holdout here with a ligand-side split (e.g. from the `scaffold` or `similarity` families) whenever both axes of leakage matter to your claim.